# MedImageInsights Backbone Verification

**Model**: `lion-ai/MedImageInsights` (Microsoft MedImageInsight, open-sourced via Lion-AI)  
**Arch**: DaViT (Dual Attention Vision Transformer) — 360M params image encoder  
**Embed dim**: 512 (projected, L2-normalized)  
**Input**: RGB 480×480 px (model's internal preprocessor; see cell 3)  
**Access**: HuggingFace snapshot download — no auth required  
**Zero-shot**: Yes — paired language encoder (252M params)  
**Modalities**: CXR, CT, MRI, dermoscopy, OCT, fundus, ultrasound, pathology, mammography

Paper: [MedImageInsight: An Open-Source Embedding Model for General Domain Medical Imaging (arXiv 2410.06542)](https://arxiv.org/abs/2410.06542)

**Training data**: 14 imaging domains (Chest X-Ray, Bone Age, Mammography, Chest CT, Breast Ultrasound,
Echocardiography, Dermoscopy, Clinical Photography, Histopathology, OCT, Fundus, and MR/CT/X-Ray exam
parameters), pooled with ~1,000 descriptive text labels and several hundred thousand free-text descriptions.
Training images were standardized to 512×512 JPEG (base64); DICOMs converted to 8-bit grayscale using
standard window levels. The deployed preprocessor resizes to **480×480** (bicubic) + ImageNet normalization.

> **Note**: The official API uses base64-encoded images. This notebook uses the official wrapper.
> To integrate into RadHarmony, bypass the base64 layer and call `classifier.model.encode_image(tensor)` directly.

In [ ]:
%pip install -q huggingface_hub Pillow torch torchvision

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Download model files from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
import os

# Downloads ~3 GB to HF cache (~/.cache/huggingface/hub/)
model_dir = snapshot_download("lion-ai/MedImageInsights")
print("Model files at:", model_dir)
print(os.listdir(model_dir))

## 2. Load via official wrapper class

In [ ]:
import sys
sys.path.insert(0, model_dir)   # makes medimageinsightmodel importable

from medimageinsightmodel import MedImageInsight

# model_dir (snapshot root) holds medimageinsightmodel.py and the MedImageInsight/ package.
# The actual weights + config.yaml live in the versioned subdirectory.
model_weights_dir = os.path.join(model_dir, "2024.09.27")

classifier = MedImageInsight(
    model_dir=model_weights_dir,
    vision_model_name="medimageinsigt-v1.0.0.pt",
    language_model_name="language_model.pth",
)
classifier.load_model()
print("Model loaded.")

## 3. Image transform (inspect what the model uses internally)

In [ ]:
# The internal transform is built from config.yaml via build_transforms()
print(classifier.preprocess)

## 3b. RadHarmony-compatible transform (draft)

Replaces the official base64 API with a MONAI-based preprocess closure that matches the other
backbone recipes. The torchvision `Resize + ToTensor + Normalize` chain from `classifier.preprocess`
is reused directly on the PIL output of the MONAI loader — no weights or config changes.

In [ ]:
import torch
import monai.transforms as mt
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

from radharmony.evaluator.backbones._utils import squeeze_frame_dim, to_3channel, to_pil_rgb

# ── Image loader: MONAI → PIL RGB ────────────────────────────────────────────
_loader = mt.Compose([
    mt.LoadImage(image_only=True, ensure_channel_first=True),
    mt.Lambda(squeeze_frame_dim),
    mt.Transpose(indices=[0, 2, 1]),   # MONAI (C,W,H) → (C,H,W)
    mt.Lambda(to_pil_rgb),             # tensor → PIL RGB (min-max → uint8)
])

# ── Mask loader: same MONAI steps, no intensity normalization ─────────────────
_loader_mask = mt.Compose([
    mt.LoadImage(image_only=True, ensure_channel_first=True),
    mt.Lambda(squeeze_frame_dim),
    mt.Transpose(indices=[0, 2, 1]),
    mt.Lambda(to_3channel),            # keep float, replicate to 3ch
])

# ── Preprocess closure: path → Tensor[3, 480, 480], ImageNet-normalized ───────
def _preprocess(path) -> torch.Tensor:
    return classifier.preprocess(_loader(str(path)))

# ── Mask preprocess: extract Resize from classifier.preprocess, swap to NEAREST
# This ensures mask spatial transform always stays in sync with the image
# transform even if config.yaml changes the target size.
_spatial_transforms = [
    T.Resize(t.size, interpolation=InterpolationMode.NEAREST, antialias=False)
    for t in classifier.preprocess.transforms
    if isinstance(t, T.Resize)
]

def _preprocess_mask(path) -> torch.Tensor:
    mask = _loader_mask(str(path))     # [3, H, W] float, class-index values
    for t in _spatial_transforms:
        mask = t(mask)
    return mask[:1]                    # [1, 480, 480]

# ── Smoke-test ────────────────────────────────────────────────────────────────
import numpy as np
from PIL import Image

dummy_rgb = Image.fromarray(np.random.randint(0, 256, (600, 800, 3), dtype=np.uint8))
dummy_rgb.save("/tmp/_mii_smoke.png")

out = _preprocess("/tmp/_mii_smoke.png")
print(f"image:  {out.shape}  dtype={out.dtype}")
print(f"range:  [{out.min():.3f}, {out.max():.3f}]")   # ImageNet-normalized

dummy_mask = Image.fromarray(np.random.randint(0, 3, (600, 800), dtype=np.uint8))
dummy_mask.save("/tmp/_mii_mask_smoke.png")
out_mask = _preprocess_mask("/tmp/_mii_mask_smoke.png")
print(f"mask:   {out_mask.shape}  dtype={out_mask.dtype}")
print(f"unique: {out_mask.unique().tolist()}")           # class indices preserved

## 4. Extract image embeddings via official API (base64 input)

In [ ]:
import base64
import io
from PIL import Image
import numpy as np

def pil_to_b64(img: Image.Image) -> str:
    """Encode a PIL image to base64 string (JPEG)."""
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

# Smoke-test with a dummy white image
dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
dummy_b64 = pil_to_b64(dummy_pil)

result = classifier.encode(images=[dummy_b64])
img_emb = result["image_embeddings"]   # numpy [B, D]
print("embedding shape:", img_emb.shape)    # expected [1, 512]
print("embedding range:", img_emb.min(), img_emb.max())
print("L2 norm:        ", np.linalg.norm(img_emb, axis=-1))   # should be ~1.0

## 5. Zero-shot: text embeddings & similarity

In [ ]:
labels = [
    "No pleural effusion.",
    "Large pleural effusion.",
    "Cardiomegaly.",
    "Normal chest X-ray.",
]

import torch
with torch.no_grad():
    result = classifier.encode(images=[dummy_b64], texts=labels)
img_emb  = result["image_embeddings"]   # [1, 512] — L2-normalized
text_emb = result["text_embeddings"]    # [4, 512] — L2-normalized

sim = (img_emb @ text_emb.T)[0]   # [4]
for lbl, s in zip(labels, sim.tolist()):
    print(f"{s:.4f}  {lbl}")

## 6. Direct tensor path (bypassing base64 — for RadHarmony integration)

In [ ]:
import torch

# The internal DaViT model is at classifier.model
# encode_image() accepts a preprocessed tensor [B, C, H, W]
device = next(classifier.model.parameters()).device

dummy_tensor = torch.rand(1, 3, 512, 512, device=device)  # after preprocess

with torch.no_grad():
    emb = classifier.model.encode_image(dummy_tensor)   # [B, 512] or similar

print("direct encode_image shape:", emb.shape)
print("type:", type(emb))
# Note: this is the raw output before CPU→numpy conversion
# For RadHarmony: model_call = lambda m, x: m.encode_image(x)

## Segmentation mode — spatial feature map

MedImageInsights uses **DaViT** (Dual Attention Vision Transformer) with 4 hierarchical stages.
Hook `image_encoder.blocks[3][0]` (the last block) to capture the final spatial feature map
`[B, H*W, C]` before global mean-pool and `avgpool`.

For 480×480 input (stride 32): **15×15** grid, **2048-dim** → `[B, 2048, 15, 15]`.

In [ ]:
import torch, math

# DaViT has 4 stages. blocks[3][0] is the last block — output [B, H*W, C] before
# norms → mean-pool → avgpool → [B, 2048].
final_feat = {}
def _hook(m, inp, out):
    t = out[0] if isinstance(out, (tuple, list)) else out
    final_feat["f"] = t

handle = classifier.model.image_encoder.blocks[3][0].register_forward_hook(_hook)
with torch.no_grad():
    _ = classifier.model.encode_image(dummy_tensor)
handle.remove()

spatial = final_feat["f"]           # [B, H*W, C]
print(f"last block output: {spatial.shape}")   # [1, 256, 2048] for 512px input

B, HW, C = spatial.shape
H = W = int(math.sqrt(HW))
assert H * W == HW, f"Non-square grid: {HW}"
spatial_map = spatial.permute(0, 2, 1).reshape(B, C, H, W)
print(f"spatial feature map: {spatial_map.shape}")   # [1, 2048, 16, 16]

# With the model's actual preprocessor (480×480): 480 / stride-32 = 15
# → RadHarmony inference shape: [B, 2048, 15, 15]

## Notes for RadHarmony integration

```
model      : classifier.model   (the DaViT nn.Module)
model_call : lambda m, x: m.encode_image(x)
pool       : None  (encode_image returns [B, 512])
embed_dim  : 512
transform  : classifier.preprocess  (Resize(480) + ToTensor + ImageNet normalize)
```

**Important**: `ImageEncoderWrapper` requires an `nn.Module` as the backbone.
`classifier.model` is a proper `nn.Module`, so `ImageEncoderWrapper.from_custom(classifier.model, model_call=...)` works.
Do NOT pass the `MedImageInsight` wrapper class itself — it's not an `nn.Module`.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Hook: `image_encoder.blocks[3][0]` (last DaViT block)
- Hook output: `[B, H*W, C]` — reshape with `permute(0,2,1).reshape(B,C,H,W)`
- For 480×480 input: **[B, 2048, 15, 15]** (DaViT stride 32)
- `torch.no_grad()` required when calling `encode_image`

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

img = Image.open(CXR_PATH).convert("RGB")
b64 = pil_to_b64(img)

finding_prompts = [
    "Findings consistent with pneumonia.",
    "No acute cardiopulmonary findings.",
    "Cardiomegaly.",
    "Pleural effusion.",
    "Pneumothorax.",
]

import torch
with torch.no_grad():
    res = classifier.encode(images=[b64], texts=finding_prompts)
sim = (res["image_embeddings"] @ res["text_embeddings"].T)[0]
for p, s in sorted(zip(finding_prompts, sim.tolist()), key=lambda x: -x[1]):
    print(f"{s:.4f}  {p}")

## Notes for RadHarmony integration

```
model      : classifier.model   (the DaViT nn.Module)
model_call : lambda m, x: m.encode_image(x)
pool       : None  (encode_image already returns [B, 512])
embed_dim  : 512
transform  : classifier.preprocess  (built from config.yaml — inspect above)
             Expected: Resize(512) + CenterCrop(512) + ToTensor +
                       Normalize(mean=..., std=...)  ← check config output in cell 3
```

**Important**: `ImageEncoderWrapper` requires an `nn.Module` as the backbone.
`classifier.model` is a proper `nn.Module`, so `ImageEncoderWrapper.from_custom(classifier.model, model_call=...)` works.
Do NOT pass the `MedImageInsight` wrapper class itself — it's not an `nn.Module`.